# Phishing Detection — ML Pipeline

**Project:** AI-Powered Phishing Email & URL Detector
**Goal of this notebook:** load the Kaggle email dataset and the PhishTank URL dataset,
explore them, extract features, and train baseline + improved ML models.

**Workflow:** Colab (train) → save model as `.pkl` → load into Django backend → serve via API → React frontend.

Run the cells **top to bottom**. Wherever you see `# >>> ADJUST` you need to check/change
something to match your actual dataset (column names, file names).

## 0. Setup — install & import libraries

In [ ]:
!pip install scikit-learn pandas numpy joblib python-whois -q

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

pd.set_option('display.max_colwidth', 100)
print("Libraries loaded.")

Libraries loaded.


## 1. Upload your datasets

Run the cell below, then click **Choose Files** and upload:
- your Kaggle phishing **email** dataset (CSV)
- your PhishTank **URL** dataset (CSV)

(If your files are already in Google Drive instead, use the Drive-mount cell below it
instead of this upload cell.)

In [ ]:
from google.colab import files
uploaded = files.upload()   # select both CSV files when prompted
print(list(uploaded.keys()))

Saving SpamEmail.csv to SpamEmail (1).csv
Saving Phishing_URL_Dataset.csv to Phishing_URL_Dataset (1).csv
['SpamEmail (1).csv', 'Phishing_URL_Dataset (1).csv']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# --- OPTIONAL: use this instead if your datasets are in Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# Then set the paths below to point inside /content/drive/MyDrive/...

In [ ]:
# >>> ADJUST these filenames to match exactly what you uploaded
EMAIL_CSV = "SpamEmail.csv"   # <-- change to your Kaggle email CSV filename
URL_CSV   = "Phishing_URL_Dataset.csv"     # <-- change to your PhishTank CSV filename

## 2. Load & inspect the EMAIL dataset

In [ ]:
email_df = pd.read_csv(EMAIL_CSV, engine='python', on_bad_lines='skip')
print("Shape:", email_df.shape)
print("\nColumns:", list(email_df.columns))
email_df.head()

Shape: (7863, 7)

Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']


,sender,receiver,date,subject,body,label,urls
0,Robert Elz <kre@munnari.OZ.AU>,Chris Garrigues <cwg-dated-1030377287.06fa6d@DeepEddy.Com>,"Thu, 22 Aug 2002 18:26:25 +0700",Re: New Sequences Window,"Date: Wed, 21 Aug 2002 10:54:46 -0500 From: Chris Garrigues Message-ID: ...",0.0,1.0
1,Steve Burt <Steve_Burt@cursor-system.com>,"""'zzzzteana@yahoogroups.com'"" <zzzzteana@yahoogroups.com>","Thu, 22 Aug 2002 12:46:18 +0100",[zzzzteana] RE: Alexander,"Martin A posted:\nTassos Papadopoulos, the Greek sculptor behind the plan, judged that the\n lim...",0.0,1.0
2,"""Tim Chapman"" <timc@2ubh.com>",zzzzteana <zzzzteana@yahoogroups.com>,"Thu, 22 Aug 2002 13:52:38 +0100",[zzzzteana] Moscow bomber,"Man Threatens Explosion In Moscow \n\nThursday August 22, 2002 1:40 PM\nMOSCOW (AP) - Security o...",0.0,1.0
3,Monty Solomon <monty@roscom.com>,undisclosed-recipient: ;,"Thu, 22 Aug 2002 09:15:25 -0400",[IRR] Klez: The Virus That Won't Die,"Klez: The Virus That Won't Die\n \nAlready the most prolific virus ever, Klez continues to wreak...",0.0,1.0
4,Stewart Smith <Stewart.Smith@ee.ed.ac.uk>,zzzzteana@yahoogroups.com,"Thu, 22 Aug 2002 14:38:22 +0100",Re: [zzzzteana] Nothing like mama used to make,"> in adding cream to spaghetti carbonara, which has the same effect on pasta as\n> making a pi...",0.0,1.0


In [ ]:
# Check class balance — run this AFTER confirming the label column name below
# >>> ADJUST: replace 'label' with your actual label column name if different
LABEL_COL = "label"   # e.g. could be 'Label', 'Email Type', 'class', etc.
print(email_df[LABEL_COL].value_counts())

label
0.0    4089
1.0    1716
Name: count, dtype: int64


## 3. Clean & prepare email text

Most Kaggle phishing-email datasets have a text column (subject/body combined or
separate) and a label column (phishing=1 / legitimate=0, or text labels like
"Phishing Email" / "Safe Email"). Adjust the column names below to match your file.

In [ ]:
# >>> ADJUST: set this to the column that holds the email text
TEXT_COL = "body"   # e.g. could be 'Email Text', 'body', 'content', etc.

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' URLTOKEN ', text)   # replace links with a token
    text = re.sub(r'[^a-z\s]', ' ', text)                    # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

email_df = email_df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
email_df['clean_text'] = email_df[TEXT_COL].apply(clean_text)

# Normalise the label to 0/1 if it's text (e.g. "Phishing Email" / "Safe Email")
if email_df[LABEL_COL].dtype == object:
    email_df['label_bin'] = email_df[LABEL_COL].str.lower().str.contains('phish').astype(int)
else:
    email_df['label_bin'] = email_df[LABEL_COL].astype(int)

email_df[['clean_text', 'label_bin']].head()



,clean_text,label_bin
0,date wed aug from chris garrigues message id tmda deepeddy vircio com i can t reproduce this err...,0
1,martin a posted tassos papadopoulos the greek sculptor behind the plan judged that the limestone...,0
2,man threatens explosion in moscow thursday august pm moscow ap security officers on thursday sei...,0
3,klez the virus that won t die already the most prolific virus ever klez continues to wreak havoc...,0
4,in adding cream to spaghetti carbonara which has the same effect on pasta as making a pizza a de...,0


In [ ]:
print(email_df['label_bin'].value_counts())

label_bin
0    4089
1    1715
Name: count, dtype: int64


## 4. Simple heuristic features (urgency words, links, etc.)

These are lightweight rule-based features that complement the ML text features
and match the "explainable heuristics" part of the project.

In [ ]:
URGENCY_WORDS = ['urgent', 'immediately', 'verify', 'suspend', 'password',
                  'click here', 'act now', 'limited time', 'confirm', 'account']

def urgency_score(text):
    text = text.lower()
    return sum(text.count(w) for w in URGENCY_WORDS)

def link_count(text):
    return len(re.findall(r'http\S+|www\S+|urltoken', text.lower()))

email_df['urgency_score'] = email_df[TEXT_COL].apply(urgency_score)
email_df['link_count'] = email_df[TEXT_COL].apply(link_count)
email_df[['clean_text', 'urgency_score', 'link_count', 'label_bin']].head()

,clean_text,urgency_score,link_count,label_bin
0,date wed aug from chris garrigues message id tmda deepeddy vircio com i can t reproduce this err...,0,1,0
1,martin a posted tassos papadopoulos the greek sculptor behind the plan judged that the limestone...,0,2,0
2,man threatens explosion in moscow thursday august pm moscow ap security officers on thursday sei...,1,2,0
3,klez the virus that won t die already the most prolific virus ever klez continues to wreak havoc...,0,2,0
4,in adding cream to spaghetti carbonara which has the same effect on pasta as making a pizza a de...,0,3,0


## 5. Train/test split + TF-IDF

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    email_df['clean_text'], email_df['label_bin'],
    test_size=0.2, random_state=42, stratify=email_df['label_bin']
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

Train shape: (4643, 5000)
Test shape: (1161, 5000)


## 6. Baseline model — Naive Bayes

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

print("=== Naive Bayes ===")
print("Accuracy :", accuracy_score(y_test, nb_preds))
print("Precision:", precision_score(y_test, nb_preds))
print("Recall   :", recall_score(y_test, nb_preds))
print("F1-score :", f1_score(y_test, nb_preds))
print("\n", classification_report(y_test, nb_preds))

=== Naive Bayes ===
Accuracy : 0.9767441860465116
Precision: 0.9619883040935673
Recall   : 0.9591836734693877
F1-score : 0.9605839416058394

               precision    recall  f1-score   support

           0       0.98      0.98      0.98       818
           1       0.96      0.96      0.96       343

    accuracy                           0.98      1161
   macro avg       0.97      0.97      0.97      1161
weighted avg       0.98      0.98      0.98      1161



## 7. Baseline model — Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)

print("=== Logistic Regression ===")
print("Accuracy :", accuracy_score(y_test, lr_preds))
print("Precision:", precision_score(y_test, lr_preds))
print("Recall   :", recall_score(y_test, lr_preds))
print("F1-score :", f1_score(y_test, lr_preds))
print("\n", classification_report(y_test, lr_preds))

=== Logistic Regression ===
Accuracy : 0.9767441860465116
Precision: 0.9817073170731707
Recall   : 0.9387755102040817
F1-score : 0.9597615499254843

               precision    recall  f1-score   support

           0       0.97      0.99      0.98       818
           1       0.98      0.94      0.96       343

    accuracy                           0.98      1161
   macro avg       0.98      0.97      0.97      1161
weighted avg       0.98      0.98      0.98      1161



## 8. Improved model — Random Forest (this is the one you'll likely deploy)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)
rf_preds = rf_model.predict(X_test_tfidf)

print("=== Random Forest ===")
print("Accuracy :", accuracy_score(y_test, rf_preds))
print("Precision:", precision_score(y_test, rf_preds))
print("Recall   :", recall_score(y_test, rf_preds))
print("F1-score :", f1_score(y_test, rf_preds))
print("\n", classification_report(y_test, rf_preds))

cm = confusion_matrix(y_test, rf_preds)
print("Confusion matrix:\n", cm)

=== Random Forest ===
Accuracy : 0.979328165374677
Precision: 0.967741935483871
Recall   : 0.9620991253644315
F1-score : 0.9649122807017544

               precision    recall  f1-score   support

           0       0.98      0.99      0.99       818
           1       0.97      0.96      0.96       343

    accuracy                           0.98      1161
   macro avg       0.98      0.97      0.98      1161
weighted avg       0.98      0.98      0.98      1161

Confusion matrix:
 [[807  11]
 [ 13 330]]


## 9. Compare all three models at a glance

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, nb_preds), accuracy_score(y_test, lr_preds), accuracy_score(y_test, rf_preds)],
    'Precision': [precision_score(y_test, nb_preds), precision_score(y_test, lr_preds), precision_score(y_test, rf_preds)],
    'Recall': [recall_score(y_test, nb_preds), recall_score(y_test, lr_preds), recall_score(y_test, rf_preds)],
    'F1-score': [f1_score(y_test, nb_preds), f1_score(y_test, lr_preds), f1_score(y_test, rf_preds)],
})
results

,Model,Accuracy,Precision,Recall,F1-score
0,Naive Bayes,0.976744,0.961988,0.959184,0.960584
1,Logistic Regression,0.976744,0.981707,0.938776,0.959762
2,Random Forest,0.979328,0.967742,0.962099,0.964912


## 10. Save the best model + the TF-IDF vectorizer

Both files are needed together — the vectorizer turns new text into the same
feature format the model was trained on. Download both `.pkl` files at the end
and place them in your Django project (e.g. inside a `ml_models/` folder).

In [ ]:
joblib.dump(rf_model, 'phishing_email_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("Saved: phishing_email_model.pkl, tfidf_vectorizer.pkl")

# Download them to your computer
from google.colab import files as colab_files
colab_files.download('phishing_email_model.pkl')
colab_files.download('tfidf_vectorizer.pkl')

Saved: phishing_email_model.pkl, tfidf_vectorizer.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 11. URL dataset — load & inspect (PhishTank)

This is a separate pipeline from the email one. Run this section once the email
part above is working.

In [ ]:
url_df = pd.read_csv(URL_CSV, engine='python', on_bad_lines='skip')
print("Shape:", url_df.shape)
print("\nColumns:", list(url_df.columns))
url_df.head()

Shape: (235795, 56)

Columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [ ]:
print(url_df[[URL_COL, URL_LABEL_COL]].head(20).to_string(index=False))

                               URL  label
  https://www.southbankmosaics.com      1
          https://www.uni-mainz.de      1
    https://www.voicefmradio.co.uk      1
       https://www.sfnmjournal.com      1
https://www.rewildingargentina.org      1
   https://www.globalreporting.org      1
        https://www.saffronart.com      1
        https://www.nerdscandy.com      1
    https://www.hyderabadonline.in      1
               https://www.aap.org      1
https://www.religionenlibertad.com      1
           http://www.teramill.com      0
      https://www.socialpolicy.org      1
             https://www.aoh61.com      1
       https://www.bulgariaski.com      1
         https://www.brightika.com      1
             https://www.motley.ie      1
            https://www.funzine.hu      1
            https://www.dixxon.com      1
           https://www.ooty.ind.in      1


In [ ]:
# >>> ADJUST these to your actual PhishTank column names
URL_COL = "URL"          # column holding the URL string
URL_LABEL_COL = "label"  # phishing=1 / legitimate=0, or similar

url_df = url_df.dropna(subset=[URL_COL, URL_LABEL_COL]).reset_index(drop=True)
if url_df[URL_LABEL_COL].dtype == object:
    url_df['url_label_bin'] = url_df[URL_LABEL_COL].str.lower().str.contains('phish').astype(int)
else:
    url_df['url_label_bin'] = url_df[URL_LABEL_COL].astype(int)

print(url_df['url_label_bin'].value_counts())

url_label_bin
1    134850
0    100945
Name: count, dtype: int64


In [ ]:
print("Original URL labels:")
print(url_df[URL_LABEL_COL].value_counts())

print("\nNumeric label mapping:")
print(url_df.groupby(URL_LABEL_COL)["url_label_bin"].value_counts())

print("\nModel class order:")
print(url_rf_model.classes_)

Original URL labels:
label
1    134850
0    100945
Name: count, dtype: int64

Numeric label mapping:
label  url_label_bin
0      0                100945
1      1                134850
Name: count, dtype: int64

Model class order:
[0 1]


## 12. URL feature extraction (lexical features)

In [ ]:
from urllib.parse import urlparse

def url_features(url):
    url = str(url)
    parsed = urlparse(url if '://' in url else 'http://' + url)
    hostname = parsed.hostname or ''
    return pd.Series({
        'url_length': len(url),
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'num_at': url.count('@'),
        'num_subdomains': hostname.count('.'),
        'has_ip': 1 if re.match(r'^\d+\.\d+\.\d+\.\d+$', hostname) else 0,
        'has_https': 1 if url.lower().startswith('https') else 0,
        'has_suspicious_word': 1 if re.search(r'login|secure|update|verify|account|bank', url.lower()) else 0,
    })

url_feat_df = url_df[URL_COL].apply(url_features)
url_feat_df['label'] = url_df['url_label_bin']
url_feat_df.head()

,url_length,num_dots,num_hyphens,num_at,num_subdomains,has_ip,has_https,has_suspicious_word,label
0,32,2,0,0,2,0,1,1,1
1,24,2,1,0,2,0,1,0,1
2,30,3,0,0,3,0,1,0,1
3,27,2,0,0,2,0,1,0,1
4,34,2,0,0,2,0,1,0,1


## 13. Train a URL classifier (Random Forest)

In [ ]:
X = url_feat_df.drop(columns=['label'])
y = url_feat_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

url_rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
url_rf_model.fit(X_train, y_train)
url_preds = url_rf_model.predict(X_test)

print("=== URL Random Forest ===")
print("Accuracy :", accuracy_score(y_test, url_preds))
print("Precision:", precision_score(y_test, url_preds))
print("Recall   :", recall_score(y_test, url_preds))
print("F1-score :", f1_score(y_test, url_preds))

# Feature importance — useful for the "explainability" part of the project
importances = pd.Series(url_rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature importance:\n", importances)

=== URL Random Forest ===
Accuracy : 0.9405627769884858
Precision: 0.9223375624759708
Recall   : 0.978457545420838
F1-score : 0.9495690973534121

Feature importance:
 has_https              0.502226
url_length             0.280836
num_hyphens            0.101652
num_subdomains         0.057016
num_dots               0.046212
has_suspicious_word    0.009536
num_at                 0.002227
has_ip                 0.000295
dtype: float64


In [ ]:
joblib.dump(url_rf_model, 'phishing_url_model.pkl')
print("Saved: phishing_url_model.pkl")

from google.colab import files as colab_files2
colab_files2.download('phishing_url_model.pkl')

Saved: phishing_url_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>